In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, random_split
import wandb
from huggingface_hub import HfApi, login
import pandas as pd
from IPython.display import display
import numpy as np
from tqdm.auto import tqdm
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

Using device: cuda
Tesla T4


In [2]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kumarmayank_25afi14 (kumarmayank_25afi14-dtu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [11]:
from google.colab import userdata
try:
    hf_token = userdata.get('')
    login(token=hf_token)
except:
    print("HF_TOKEN secret not found. Please login interactively:")
    login()

HF_TOKEN secret not found. Please login interactively:


In [4]:
batch_size = 256
transform_base = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])
transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])
full_trainset = datasets.CIFAR10(root='./data', train=True, download=True)
full_testset = datasets.CIFAR10(root='./data', train=False, download=True)
combined_data = torch.utils.data.ConcatDataset([full_trainset, full_testset])

total_size = len(combined_data)
train_size = int(0.8 * total_size)
val_size = int(0.1 * total_size)
test_size = total_size - train_size - val_size

generator = torch.Generator().manual_seed(42)
train_idx, val_idx, test_idx = random_split(range(total_size), [train_size, val_size, test_size], generator=generator)

class CIFARSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y
    def __len__(self):
        return len(self.subset)
train_data_base = CIFARSubset(Subset(combined_data, train_idx), transform=transform_base)
train_data_aug = CIFARSubset(Subset(combined_data, train_idx), transform=transform_aug)
val_data = CIFARSubset(Subset(combined_data, val_idx), transform=transform_base)
test_data = CIFARSubset(Subset(combined_data, test_idx), transform=transform_base)

loader_train_base = DataLoader(train_data_base, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
loader_train_aug = DataLoader(train_data_aug, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
loader_val = DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
loader_test = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
print(f"Dataset split -> Train: {len(train_data_base)} | Val: {len(val_data)} | Test: {len(test_data)}")

100%|██████████| 170M/170M [00:03<00:00, 46.2MB/s] 


Dataset split -> Train: 48000 | Val: 6000 | Test: 6000


In [5]:
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels=3, patch_size=4, embed_dim=128, img_size=32):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x).flatten(2).transpose(1, 2)
        return x

class ViT(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3, num_classes=10, 
                 embed_dim=128, depth=4, num_heads=4, mlp_ratio=2.0):
        super().__init__()
        self.patch_embed = PatchEmbedding(in_channels, patch_size, embed_dim, img_size)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        
        self.pos_embed = nn.Parameter(torch.zeros(1, self.patch_embed.num_patches + 1, embed_dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=int(embed_dim * mlp_ratio),
            dropout=0.1, activation="gelu", batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(embed_dim)

        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed
        x = self.transformer(x)
        cls_out = self.norm(x[:, 0])
        return self.head(cls_out)

def get_resnet18():
    model = torchvision.models.resnet18(weights=None)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, 10)
    return model

In [6]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(reduction='none')

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

def get_criterion(loss_name):
    if loss_name == "CrossEntropy":
        return nn.CrossEntropyLoss()
    elif loss_name == "LabelSmoothing":
        return nn.CrossEntropyLoss(label_smoothing=0.1)
    elif loss_name == "FocalLoss":
        return FocalLoss(alpha=1.0, gamma=2.0)

In [7]:
def train(model_name, use_aug, loss_name, opt_name, epochs=5):
    model = ViT().to(device) if model_name == "ViT" else get_resnet18().to(device)
    train_loader = loader_train_aug if use_aug else loader_train_base

    criterion = get_criterion(loss_name)
    lr = 1e-3
    if opt_name == "Adam": optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    elif opt_name == "RMSprop": optimizer = torch.optim.RMSprop(model.parameters(), lr=lr, momentum=0.0)
    elif opt_name == "SGD": optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.0)

    run = wandb.init(project="dl-lab-10-vit", 
                     name=f"{model_name}_Aug{use_aug}_{loss_name}_{opt_name}", 
                     config={"model": model_name, "aug": use_aug, "loss": loss_name, "opt": opt_name},)
    
    print(f"{model_name}_Aug{use_aug}_{loss_name}_{opt_name}")
    for epoch in range(epochs):
        model.train()
        train_loss, correct, total = 0, 0, 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        
        for inputs, targets in pbar:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            
        train_acc = 100. * correct / total
        avg_train_loss = train_loss / len(train_loader)
        
        model.eval()
        val_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for inputs, targets in loader_val:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()
                
        val_acc = 100. * correct / total
        avg_val_loss = val_loss / len(loader_val)
        
        wandb.log({"train_loss": avg_train_loss, "train_acc": train_acc, 
                   "val_loss": avg_val_loss, "val_acc": val_acc})

    model.eval()
    test_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for inputs, targets in loader_test:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            
    test_acc = 100. * correct / total
    avg_test_loss = test_loss / len(loader_test)
    wandb.log({"test_loss": avg_test_loss, "test_acc": test_acc})
    
    run.finish()
    
    return model, avg_train_loss, train_acc, avg_val_loss, val_acc, avg_test_loss, test_acc

In [8]:
augmentation = [True, False]
losses = ["CrossEntropy", "LabelSmoothing", "FocalLoss"]
optimizers = ["Adam", "RMSprop", "SGD"]
EPOCHS = 50
results = []
trained_models = {}

for aug in augmentation:
    for loss in losses:
        for opt in optimizers:
            model, t_loss, t_acc, v_loss, v_acc, te_loss, te_acc = train(model_name="ViT", use_aug=aug, loss_name=loss, opt_name=opt, epochs=EPOCHS)
            
            results.append({
                "Model": "ViT",
                "Augmented": aug,
                "Loss Function": loss,
                "Optimizer": opt,
                "Train Loss": round(t_loss, 4),
                "Train Acc (%)": round(t_acc, 2),
                "Val Loss": round(v_loss, 4),
                "Val Acc (%)": round(v_acc, 2),
                "Test Loss": round(te_loss, 4),
                "Test Acc (%)": round(te_acc, 2)
            })
            model_name = f"ViT_Aug{aug}_{loss}_{opt}"
            trained_models[model_name] = model

df_results = pd.DataFrame(results)
display(df_results)

ViT_AugTrue_CrossEntropy_Adam


Epoch 1/50:   0%|          | 0/188 [00:00<?, ?it/s]

Epoch 2/50:   0%|          | 0/188 [00:00<?, ?it/s]

Epoch 3/50:   0%|          | 0/188 [00:00<?, ?it/s]

Epoch 4/50:   0%|          | 0/188 [00:00<?, ?it/s]

...

Epoch 48/50:   0%|          | 0/188 [00:00<?, ?it/s]

Epoch 49/50:   0%|          | 0/188 [00:00<?, ?it/s]

Epoch 50/50:   0%|          | 0/188 [00:00<?, ?it/s]

test_acc,▁
test_loss,▁
train_acc,▁▂▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇████████
train_loss,█▆▆▆▅▅▅▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
val_acc,▁▂▂▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇██████
val_loss,█▇▇▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁
test_acc,45.36667
test_loss,1.01281
train_acc,44.18958
train_loss,1.04061
val_acc,45.11667


,Model,Augmented,Loss Function,Optimizer,Train Loss,Train Acc (%),Val Loss,Val Acc (%),Test Loss,Test Acc (%)
0,ViT,True,CrossEntropy,Adam,0.4659,83.12,1.0052,70.57,0.9757,70.38
1,ViT,True,CrossEntropy,RMSprop,0.6863,75.20,1.1322,63.33,1.1272,62.82
2,ViT,True,CrossEntropy,SGD,1.6194,41.05,1.5793,43.10,1.5694,42.58
3,ViT,True,LabelSmoothing,Adam,0.8922,82.48,1.2380,69.98,1.2427,69.52
4,ViT,True,LabelSmoothing,RMSprop,1.0502,75.22,1.4205,61.20,1.4110,61.05
5,ViT,True,LabelSmoothing,SGD,1.8235,37.26,1.7981,38.33,1.8028,38.35
6,ViT,True,FocalLoss,Adam,0.2528,79.35,0.6165,67.98,0.5965,67.68
7,ViT,True,FocalLoss,RMSprop,0.3728,72.46,0.7089,61.90,0.6872,60.82
8,ViT,True,FocalLoss,SGD,1.1277,39.71,1.1006,41.20,1.0975,41.78
9,ViT,False,CrossEntropy,Adam,0.2573,90.73,1.3844,67.45,1.3958,67.23


In [9]:
# Parameters for Augmented=True
best_loss_aug = "CrossEntropy"
best_opt_aug = "Adam"

# Parameters for Augmented=False
best_loss_base = "FocalLoss"
best_opt_base = "Adam"

resnet_results = []

resnetModelAug, t_loss, t_acc, v_loss, v_acc, te_loss, te_acc = train_and_evaluate(model_name="ResNet-18", use_aug=True, loss_name=best_loss_aug, opt_name=best_opt_aug, epochs=EPOCHS)
resnet_results.append({
    "Model": "ResNet-18", "Augmented": True, "Loss Function": best_loss_aug, "Optimizer": best_opt_aug,
    "Train Loss": round(t_loss, 4), "Train Acc (%)": round(t_acc, 2),
    "Val Loss": round(v_loss, 4), "Val Acc (%)": round(v_acc, 2),
    "Test Loss": round(te_loss, 4), "Test Acc (%)": round(te_acc, 2)
})
model_name = f"ResNet18_AugTrue_{best_loss_aug}_{best_opt_aug}"
trained_models[model_name] = resnetModelAug

resnetModel, t_loss, t_acc, v_loss, v_acc, te_loss, te_acc = train_and_evaluate(model_name="ResNet-18", use_aug=False, loss_name=best_loss_base, opt_name=best_opt_base, epochs=EPOCHS)
resnet_results.append({
    "Model": "ResNet-18", "Augmented": False, "Loss Function": best_loss_base, "Optimizer": best_opt_base,
    "Train Loss": round(t_loss, 4), "Train Acc (%)": round(t_acc, 2),
    "Val Loss": round(v_loss, 4), "Val Acc (%)": round(v_acc, 2),
    "Test Loss": round(te_loss, 4), "Test Acc (%)": round(te_acc, 2)
})
model_name = f"ResNet18_AugFalse_{best_loss_base}_{best_opt_base}"
trained_models[model_name] = resnetModel

df_resnet = pd.DataFrame(resnet_results)
display(df_resnet)

ResNet-18_AugTrue_CrossEntropy_Adam


Epoch 1/50:   0%|          | 0/188 [00:00<?, ?it/s]

Epoch 2/50:   0%|          | 0/188 [00:00<?, ?it/s]

Epoch 3/50:   0%|          | 0/188 [00:00<?, ?it/s]

Epoch 4/50:   0%|          | 0/188 [00:00<?, ?it/s]

...

Epoch 48/50:   0%|          | 0/188 [00:00<?, ?it/s]

Epoch 49/50:   0%|          | 0/188 [00:00<?, ?it/s]

Epoch 50/50:   0%|          | 0/188 [00:00<?, ?it/s]

test_acc,▁
test_loss,▁
train_acc,▁▄▅▆▆▇▇▇████████████████████████████████
train_loss,█▆▄▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▅▄▇▆▇▇▇▇▇█▇▇█▇▇█▇▇▇▇▇▇█▇██▇▇██▇████████
val_loss,█▃▁▃▃▂▃▅▂▃▄▂▄▃▃▅▂▂▄▄▃▄▃▄▃▃▃▃▃▄▃▅▃▃▃▄▄▄▄▃
test_acc,83.7
test_loss,0.46085
train_acc,99.30208
train_loss,0.00651
val_acc,82.63333


,Model,Augmented,Loss Function,Optimizer,Train Loss,Train Acc (%),Val Loss,Val Acc (%),Test Loss,Test Acc (%)
0,ResNet-18,True,CrossEntropy,Adam,0.0405,98.57,0.7192,85.52,0.6798,86.05
1,ResNet-18,False,FocalLoss,Adam,0.0065,99.30,0.4957,82.63,0.4609,83.70


In [13]:
repo_id = "MK80310/DLLab10" 
api = HfApi()
try:
    api.create_repo(repo_id=repo_id, exist_ok=True)
    print(f"Repository {repo_id} ready.")
    for name, model in trained_models.items():
        save_path = f"{name}.pth"
        torch.save(model.state_dict(), save_path)
        
        api.upload_file(
            path_or_fileobj=save_path,
            path_in_repo=f"models/{save_path}",
            repo_id=repo_id,
            commit_message=f"Add {name} weights"
        )        
    print("\nAll models successfully pushed to Hugging Face Hub!")
    
except Exception as e:
    print(f"An error occurred while uploading to Hugging Face: {e}")

Repository MK80310/DLLab10 ready.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


All models successfully pushed to Hugging Face Hub!


In [15]:
data = [
    ["ViT", False, "FocalLoss", "Adam", 0.1220, 88.02, 0.6812, 68.78, 0.7045, 68.20],
    ["ResNet-18", False, "FocalLoss", "Adam", 0.0065, 99.30, 0.4957, 82.63, 0.4609, 83.70],

    ["ViT", True, "CrossEntropy", "Adam", 0.4659, 83.12, 1.0052, 70.57, 0.9757, 70.38],
    ["ResNet-18", True, "CrossEntropy", "Adam", 0.0405, 98.57, 0.7192, 85.52, 0.6798, 86.05],
    
]

columns = [
    "Model", "Augmented", "Loss Function", "Optimizer",
    "Train Loss", "Train Acc (%)",
    "Val Loss", "Val Acc (%)",
    "Test Loss", "Test Acc (%)"
]

comparison = pd.DataFrame(data, columns=columns)
display(comparison)

,Model,Augmented,Loss Function,Optimizer,Train Loss,Train Acc (%),Val Loss,Val Acc (%),Test Loss,Test Acc (%)
0,ViT,False,FocalLoss,Adam,0.1220,88.02,0.6812,68.78,0.7045,68.20
1,ResNet-18,False,FocalLoss,Adam,0.0065,99.30,0.4957,82.63,0.4609,83.70
2,ViT,True,CrossEntropy,Adam,0.4659,83.12,1.0052,70.57,0.9757,70.38
3,ResNet-18,True,CrossEntropy,Adam,0.0405,98.57,0.7192,85.52,0.6798,86.05
